In [ ]:
import os
os.environ['PYSPARK_PYTHON'] = '/usr/bin/python3.9'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/usr/bin/python3.9'

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Word Count - Pan Tadeusz") \
    .master("yarn") \
    .config("spark.cores.max", "4") \
    .config("spark.sql.shuffle.partitions", "5") \
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3.9") \
    .config("spark.executor.extraJavaOptions", "-XX:-UseContainerSupport") \
    .config("spark.driver.extraJavaOptions", "-XX:-UseContainerSupport") \
    .getOrCreate()

input_data = "hdfs://master:9000/pan_tadeusz.txt" 
output_data = "hdfs://master:9000/output/pan_tadeusz_count"

lines = spark.sparkContext.textFile(input_data)

out = lines.flatMap(lambda line: line.lower().split()) \
           .map(lambda word: (word, 1)) \
           .reduceByKey(lambda a, b: a + b)

out.saveAsTextFile(output_data)

print(f"Zadanie zakończone! Wyniki zapisano na HDFS: {output_data}")

spark.stop()